# 05 — Feature engineering

This notebook turns canonical PON telemetry into causal, metric-aware model
features. It uses the catalogue and compact time-window policy frozen before
modelling: gauges preserve their units, exact lags represent 1-hour and
6-hour change, robust history represents 24-hour and 7-day behaviour,
zero-inflated BER and CRC values use hurdle features, counts
use non-negative transforms, and cumulative counters are differenced without
bridging resets or collection gaps.

Only calibration and development telemetry are opened. No fault, ticket or
holdout label is read.


## 1. Setup and frozen inputs


In [ ]:
from pathlib import Path
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    """Find the checked-out repository when Jupyter starts in any subfolder."""
    override = os.getenv("TELCO_PROJECT_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents]
        if "google.colab" in sys.modules:
            candidates += [
                Path("/content/drive/MyDrive/anomaly_detection"),
                Path("/content/drive/MyDrive/telco-anomaly-detection"),
            ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, or set TELCO_PROJECT_ROOT."
    )


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import shutil
import tempfile

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

from telco_anomaly.detectors import (
    iter_episode_frames,
    materialize_measurement_features,
    materialize_wide_partition,
    measurement_features,
    split_feature_file_by_time,
)
from telco_anomaly.features import feature_policy
from telco_anomaly.io import (
    file_sha256,
    immutable_output_directory,
    load_config,
    read_json,
    require_same,
    resolve_data_root,
    write_json,
)

DATA_ROOT = resolve_data_root()
DATASET = os.getenv("TELCO_DATASET", "synthetic_pon")
default_core_runs = {
    "synthetic_pon": "synthetic_pon_core_v2",
    "ran_pm": "ran_pm_v1",
    "microsoft_optical": "microsoft_optical_v1",
}
if DATASET not in default_core_runs:
    raise ValueError(f"Feature engineering is not configured for {DATASET!r}")
legacy_core = os.getenv("PON_CORE_RUN_ID") if DATASET == "synthetic_pon" else None
legacy_eda = os.getenv("PON_EDA_RUN_ID") if DATASET == "synthetic_pon" else None
legacy_features = os.getenv("PON_FEATURE_RUN_ID") if DATASET == "synthetic_pon" else None
CORE_RUN_ID = os.getenv("TELCO_CORE_RUN_ID", legacy_core or default_core_runs[DATASET])
EDA_RUN_ID = os.getenv(
    "TELCO_EDA_RUN_ID", legacy_eda or f"{DATASET}_calibration_eda_v3"
)
FEATURE_RUN_ID = os.getenv(
    "TELCO_FEATURE_RUN_ID", legacy_features or f"{DATASET}_features_v4"
)

RUN_ROOT = DATA_ROOT / "core" / DATASET / CORE_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
SPLIT_ROOT = RUN_ROOT / "SPLITS"
EDA_ROOT = DATA_ROOT / "eda" / DATASET / EDA_RUN_ID
OUTPUT_ROOT = DATA_ROOT / "features" / DATASET / FEATURE_RUN_ID

for required in (CORE_ROOT, SPLIT_ROOT, EDA_ROOT):
    if not required.exists():
        raise FileNotFoundError(f"Missing prerequisite {required}")
if (RUN_ROOT / "SPEC-EVAL").exists():
    raise PermissionError("Feature engineering must use the truth-unmounted run")

catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
core_manifest = read_json(CORE_ROOT / "manifest.json")
eda_manifest = read_json(EDA_ROOT / "eda_manifest.json")
eda_decisions = read_json(EDA_ROOT / "eda_decisions.json")
assert eda_manifest["partition"] == "calibration"
assert eda_manifest["core_fingerprint"] == core_manifest["fingerprint"]
assert eda_decisions["canonical_fingerprint"] == core_manifest["fingerprint"]

FEATURE_CONFIG = load_config("features", project_root=PROJECT_ROOT)
TEMPORAL = FEATURE_CONFIG["temporal"]
HISTORY_WINDOWS_SECONDS = TEMPORAL["history_windows_seconds"]
LAG_WINDOWS_SECONDS = TEMPORAL["lag_windows_seconds"]
ACTIVITY_WINDOWS_SECONDS = TEMPORAL["activity_windows_seconds"]
HISTORY_METRIC_IDS = TEMPORAL["history_metric_ids"]
LAG_METRIC_IDS = TEMPORAL["lag_metric_ids"]
ACTIVITY_METRIC_IDS = TEMPORAL["activity_metric_ids"]
MINIMUM_WINDOW_FRACTION = float(TEMPORAL["minimum_window_fraction"])
DISPERSION_WINDOW_SECONDS = int(eda_decisions["dispersion_window_seconds"])
SEASONAL_PERIODS = {
    metric_id: period
    for metric_id, period in eda_decisions["seasonality_decisions"].items()
    if period is not None
}
GAP_TOLERANCE = float(eda_decisions.get("gap_tolerance", 1.5))
LOOKBACK_SECONDS = max(
    *HISTORY_WINDOWS_SECONDS.values(),
    *LAG_WINDOWS_SECONDS.values(),
    *ACTIVITY_WINDOWS_SECONDS.values(),
    *SEASONAL_PERIODS.values(),
)
CALIBRATION_FIT_FRACTION = 0.70
SCRATCH_PARENT = Path(os.getenv(
    "TELCO_WORK_ROOT", "/content" if "google.colab" in sys.modules else tempfile.gettempdir()
))
SCRATCH_PARENT.mkdir(parents=True, exist_ok=True)

display(pd.Series({
    "canonical_input": str(CORE_ROOT),
    "dataset": DATASET,
    "calibration_eda": str(EDA_ROOT),
    "feature_output": str(OUTPUT_ROOT),
    "history_windows": HISTORY_WINDOWS_SECONDS,
    "lag_windows": LAG_WINDOWS_SECONDS,
    "activity_windows": ACTIVITY_WINDOWS_SECONDS,
    "history_metrics": HISTORY_METRIC_IDS,
    "lag_metrics": LAG_METRIC_IDS,
    "activity_metrics": ACTIVITY_METRIC_IDS,
    "dispersion_hours": DISPERSION_WINDOW_SECONDS / 3600,
    "seasonal_metrics": len(SEASONAL_PERIODS),
    "calibration_fit_fraction": CALIBRATION_FIT_FRACTION,
    "canonical_long_rows": core_manifest["row_counts"]["telemetry"],
    "progress_reporting": "pivot completion, then every 25 episodes",
    "local_scratch": str(SCRATCH_PARENT),
}, name="value").to_frame())


## 2. Inspect the feature policy

The table below is the auditable bridge from semantic metrics to features.
`asset_health` features may enter a detector. Clipping flags remain
`data_quality`; they are retained for monitoring but cannot become health
evidence.


In [ ]:
policy = feature_policy(catalogue)
display(policy)

assert set(policy.loc[policy["feature"].str.endswith("__clipped"), "role"]) == {
    "data_quality"
}
assert not policy["feature"].str.startswith(("gt_", "truth_", "fault_"), na=False).any()


## 3. Materialise features with bounded local storage

The full calibration and development partitions are processed; this is not a
small analytical sample. Large temporary wide panels stay on the runtime's
local disk, progress is printed every 25 entity episodes, and only final
feature files are copied to persistent storage. Calibration is split
chronologically: the early slice fits the detector and the disjoint late slice
sets score thresholds.


In [ ]:
def build_partition(partition, scratch):
    wide_path = scratch / f"{partition}_wide.parquet"
    feature_path = scratch / f"{partition}_features.parquet"
    print(f"\nStarting {partition}", flush=True)
    definition = materialize_wide_partition(
        CORE_ROOT,
        SPLIT_ROOT,
        partition,
        catalogue,
        wide_path,
        lookback_seconds=LOOKBACK_SECONDS,
        memory_limit=os.getenv("DUCKDB_MEMORY_LIMIT", "2GB"),
        threads=int(os.getenv("DUCKDB_THREADS", "2")),
    )
    materialize_measurement_features(
        wide_path,
        catalogue,
        feature_path,
        gap_tolerance=GAP_TOLERANCE,
        seasonal_periods=SEASONAL_PERIODS,
        history_windows_seconds=HISTORY_WINDOWS_SECONDS,
        lag_windows_seconds=LAG_WINDOWS_SECONDS,
        activity_windows_seconds=ACTIVITY_WINDOWS_SECONDS,
        history_metric_ids=HISTORY_METRIC_IDS,
        lag_metric_ids=LAG_METRIC_IDS,
        activity_metric_ids=ACTIVITY_METRIC_IDS,
        minimum_window_fraction=MINIMUM_WINDOW_FRACTION,
        score_start=definition["score_start"],
        score_end=definition["score_end"],
        progress_every=25,
    )
    return wide_path, feature_path, definition


EDA_DECISIONS_SHA256 = file_sha256(EDA_ROOT / "eda_decisions.json")
TIME_PARTITIONS_SHA256 = file_sha256(SPLIT_ROOT / "time_partitions.parquet")
FEATURES_MODULE_SHA256 = file_sha256(
    PROJECT_ROOT / "src" / "telco_anomaly" / "features.py"
)
DETECTORS_MODULE_SHA256 = file_sha256(
    PROJECT_ROOT / "src" / "telco_anomaly" / "detectors.py"
)
FEATURE_CONFIG_SHA256 = file_sha256(PROJECT_ROOT / "configs" / "features.yml")
expected_inputs = {
    "core_fingerprint": core_manifest["fingerprint"],
    "eda_decisions_sha256": EDA_DECISIONS_SHA256,
    "time_partitions_sha256": TIME_PARTITIONS_SHA256,
    "features_module_sha256": FEATURES_MODULE_SHA256,
    "detectors_module_sha256": DETECTORS_MODULE_SHA256,
    "feature_config_sha256": FEATURE_CONFIG_SHA256,
    "history_windows_seconds": HISTORY_WINDOWS_SECONDS,
    "lag_windows_seconds": LAG_WINDOWS_SECONDS,
    "activity_windows_seconds": ACTIVITY_WINDOWS_SECONDS,
    "history_metric_ids": HISTORY_METRIC_IDS,
    "lag_metric_ids": LAG_METRIC_IDS,
    "activity_metric_ids": ACTIVITY_METRIC_IDS,
    "minimum_window_fraction": MINIMUM_WINDOW_FRACTION,
    "dispersion_window_seconds": DISPERSION_WINDOW_SECONDS,
    "seasonal_periods": SEASONAL_PERIODS,
    "gap_tolerance": GAP_TOLERANCE,
    "calibration_fit_fraction": CALIBRATION_FIT_FRACTION,
}

if OUTPUT_ROOT.exists():
    feature_manifest = read_json(OUTPUT_ROOT / "feature_manifest.json")
    require_same(feature_manifest, **expected_inputs)
    print("Using existing immutable features:", OUTPUT_ROOT)
else:
    with immutable_output_directory(OUTPUT_ROOT) as output:
        with tempfile.TemporaryDirectory(
            dir=SCRATCH_PARENT, prefix="telco-features-"
        ) as scratch_name:
            scratch = Path(scratch_name)
            calibration_wide, calibration_features, _ = build_partition(
                "calibration", scratch
            )
            causality_sample = next(iter_episode_frames(calibration_wide))
            causality_sample.to_parquet(
                output / "causality_sample.parquet", index=False
            )

            fit_local = scratch / "calibration_fit_features.parquet"
            threshold_local = scratch / "calibration_threshold_features.parquet"
            split = split_feature_file_by_time(
                calibration_features, fit_local, threshold_local,
                fit_fraction=CALIBRATION_FIT_FRACTION,
            )
            files = {
                "calibration_fit": fit_local,
                "calibration_threshold": threshold_local,
            }
            partitions = {}
            for name, source in files.items():
                destination = output / f"{name}_features.parquet"
                shutil.copy2(source, destination)
                partitions[name] = {
                    "features": destination.name,
                    "rows": pq.ParquetFile(destination).metadata.num_rows,
                }
            calibration_wide.unlink()
            calibration_features.unlink()
            fit_local.unlink()
            threshold_local.unlink()

            development_wide, development_local, _ = build_partition(
                "development", scratch
            )
            destination = output / "development_features.parquet"
            shutil.copy2(development_local, destination)
            partitions["development"] = {
                "features": destination.name,
                "rows": pq.ParquetFile(destination).metadata.num_rows,
            }

        generated_columns = pq.ParquetFile(
            output / partitions["calibration_fit"]["features"]
        ).schema_arrow.names
        generated_features = [
            name for name in generated_columns
            if name not in {"event_ts", "entity_id", "episode_id"}
        ]
        generated_policy = feature_policy(catalogue, generated_features)
        generated_policy.to_parquet(
            output / "feature_policy.parquet", index=False
        )
        feature_manifest = {
            "dataset": DATASET,
            **expected_inputs,
            "eda_manifest_sha256": file_sha256(EDA_ROOT / "eda_manifest.json"),
            "calibration_threshold_start": split["cutoff"],
            "partitions": partitions,
            "truth_files_read": [],
            "holdout_materialised": False,
        }
        write_json(output / "feature_manifest.json", feature_manifest)
    print("Saved:", OUTPUT_ROOT)

display(pd.Series(feature_manifest["partitions"], name="partition evidence").to_frame())


## 4. Causality test

For one complete calibration episode, features calculated on a time prefix
must equal the corresponding prefix calculated when later rows are present.
This catches centred windows, backward filling and other future leakage.


In [ ]:
episode = pd.read_parquet(OUTPUT_ROOT / "causality_sample.parquet")
if len(episode) < 20:
    raise ValueError("A calibration episode is too short for the causality test")

cutoff = max(10, len(episode) // 2)
full = measurement_features(
    episode,
    catalogue,
    gap_tolerance=GAP_TOLERANCE,
    seasonal_periods=SEASONAL_PERIODS,
    history_windows_seconds=HISTORY_WINDOWS_SECONDS,
    lag_windows_seconds=LAG_WINDOWS_SECONDS,
    activity_windows_seconds=ACTIVITY_WINDOWS_SECONDS,
    history_metric_ids=HISTORY_METRIC_IDS,
    lag_metric_ids=LAG_METRIC_IDS,
    activity_metric_ids=ACTIVITY_METRIC_IDS,
    minimum_window_fraction=MINIMUM_WINDOW_FRACTION,
).iloc[:cutoff].reset_index(drop=True)
prefix = measurement_features(
    episode.iloc[:cutoff].copy(),
    catalogue,
    gap_tolerance=GAP_TOLERANCE,
    seasonal_periods=SEASONAL_PERIODS,
    history_windows_seconds=HISTORY_WINDOWS_SECONDS,
    lag_windows_seconds=LAG_WINDOWS_SECONDS,
    activity_windows_seconds=ACTIVITY_WINDOWS_SECONDS,
    history_metric_ids=HISTORY_METRIC_IDS,
    lag_metric_ids=LAG_METRIC_IDS,
    activity_metric_ids=ACTIVITY_METRIC_IDS,
    minimum_window_fraction=MINIMUM_WINDOW_FRACTION,
).reset_index(drop=True)

pd.testing.assert_frame_equal(full, prefix, check_exact=True)
print("PASS — later observations cannot change earlier feature values")


## 5. Feature sample and acceptance


In [ ]:
feature_path = OUTPUT_ROOT / feature_manifest["partitions"]["calibration_fit"]["features"]
sample = next(pq.ParquetFile(feature_path).iter_batches(batch_size=10)).to_pandas()
display(sample)

generated_policy = pd.read_parquet(OUTPUT_ROOT / "feature_policy.parquet")
available = generated_policy.loc[
    generated_policy["role"].eq("asset_health"), "feature"
].sort_values().tolist()
display(generated_policy)

assert feature_manifest["truth_files_read"] == []
assert feature_manifest["holdout_materialised"] is False
assert available
print("PASS — causal model features are ready")
print("Next: 06_PRIMARY_UNSUPERVISED_MODEL.ipynb")
